In [2]:

import json
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    log_loss,
    brier_score_loss,
    roc_auc_score,
)

from cricketwp.predict import chase_curve


# ============================================================
# 1. LOCATE PROJECT ROOT
# ============================================================

ROOT = Path.cwd()

# Notebook may be running from the notebooks/ directory
if not (ROOT / "data" / "processed").exists():
    ROOT = ROOT.parent

assert (ROOT / "data" / "processed" / "chase_test.parquet").exists(), (
    f"Could not find chase_test.parquet. Detected root: {ROOT}"
)

print("PROJECT ROOT:", ROOT)


# ============================================================
# 2. LOAD DATASETS
# ============================================================

train = pd.read_parquet(
    ROOT / "data" / "processed" / "chase_train.parquet"
)

te = pd.read_parquet(
    ROOT / "data" / "processed" / "chase_test.parquet"
)

print("\nDATASET INFORMATION")
print("Training rows:", len(train))
print("Test rows:", len(te))
print("Training matches:", train["match_id"].nunique())
print("Test matches:", te["match_id"].nunique())


# ============================================================
# 3. LOAD PRODUCTION MODEL
# ============================================================

clf = xgb.XGBClassifier()

clf.load_model(
    str(ROOT / "artifacts" / "chase_xgb.json")
)

with open(
    ROOT / "artifacts" / "chase_iso.json",
    encoding="utf-8",
) as f:
    iso = json.load(f)

iso_x = np.asarray(iso["x"])
iso_y = np.asarray(iso["y"])

with open(
    ROOT / "artifacts" / "meta.json",
    encoding="utf-8",
) as f:
    meta = json.load(f)

print("\nMODEL VERSION:", meta["version"])


# ============================================================
# 4. REPRODUCE PRODUCTION TEST PREDICTIONS
# ============================================================

curve = chase_curve(
    te,
    clf,
    iso_x,
    iso_y,
)

p_test = curve["p_chase"].to_numpy()
y_test = te["label"].to_numpy()

assert len(p_test) == len(y_test)
assert np.isfinite(p_test).all()
assert np.isin(y_test, [0, 1]).all()

print("\nPREDICTION CHECKS")
print("Predictions:", len(p_test))
print("Minimum probability:", p_test.min())
print("Maximum probability:", p_test.max())
print("NaN predictions:", np.isnan(p_test).sum())


# ============================================================
# 5. RECOMPUTE TEST METRICS
# ============================================================

reproduced = {
    "logloss": log_loss(
        y_test,
        p_test,
        labels=[0, 1],
    ),
    "brier": brier_score_loss(
        y_test,
        p_test,
    ),
    "auc": roc_auc_score(
        y_test,
        p_test,
    ),
}

print("\n" + "=" * 65)
print("REPRODUCED MODEL TEST METRICS")
print("=" * 65)

for name, value in reproduced.items():
    print(f"{name:12s}: {value:.8f}")


# ============================================================
# 6. COMPARE AGAINST SAVED METRICS
# ============================================================

saved = meta["test_metrics"]

print("\n" + "=" * 65)
print("SAVED VS REPRODUCED METRICS")
print("=" * 65)

for name, value in reproduced.items():
    original = saved[name]
    difference = abs(value - original)

    print(
        f"{name:12s} | "
        f"Saved: {original:.8f} | "
        f"Reproduced: {value:.8f} | "
        f"Difference: {difference:.10f}"
    )


# ============================================================
# 7. PREPARE BASELINE REFERENCE
# ============================================================

# Simple baseline:
# Always predict the chase-win frequency observed in training.

baseline_probability = float(train["label"].mean())

baseline_predictions = np.full(
    len(y_test),
    baseline_probability,
)

baseline_scores = {
    "logloss": log_loss(
        y_test,
        baseline_predictions,
        labels=[0, 1],
    ),
    "brier": brier_score_loss(
        y_test,
        baseline_predictions,
    ),
    "auc": 0.5,
}

print("\n" + "=" * 65)
print("SIMPLE BASELINE")
print("=" * 65)

print(
    "Training chase-win frequency:",
    round(baseline_probability, 6),
)

for name, value in baseline_scores.items():
    print(f"{name:12s}: {value:.8f}")


# ============================================================
# 8. FINAL EVALUATION SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("TRUST EVALUATION SUMMARY")
print("=" * 65)

print("Test rows:", len(te))
print("Unique test matches:", te["match_id"].nunique())
print("Actual chase-win rate:", round(float(y_test.mean()), 6))
print("Baseline probability:", round(baseline_probability, 6))

print("\nMetric reproduction status:")

for name, value in reproduced.items():
    difference = abs(value - saved[name])
    status = "MATCH" if difference < 1e-5 else "CHECK"

    print(f"{name:12s}: {status}")

print("\nStep 1 complete.")


PROJECT ROOT: c:\Users\karne_dvob0bk\cric-win-pred

DATASET INFORMATION
Training rows: 302180
Test rows: 114440
Training matches: 2811
Test matches: 1133

MODEL VERSION: 1.0.0

PREDICTION CHECKS
Predictions: 114440
Minimum probability: 0.0
Maximum probability: 1.0
NaN predictions: 0

REPRODUCED MODEL TEST METRICS
logloss     : 0.32540718
brier       : 0.10287301
auc         : 0.93767979

SAVED VS REPRODUCED METRICS
logloss      | Saved: 0.32171693 | Reproduced: 0.32540718 | Difference: 0.0036902432
brier        | Saved: 0.10287301 | Reproduced: 0.10287301 | Difference: 0.0000000003
auc          | Saved: 0.93767979 | Reproduced: 0.93767979 | Difference: 0.0000000000

SIMPLE BASELINE
Training chase-win frequency: 0.491432
logloss     : 0.69233922
brier       : 0.24959606
auc         : 0.50000000

TRUST EVALUATION SUMMARY
Test rows: 114440
Unique test matches: 1133
Actual chase-win rate: 0.472143
Baseline probability: 0.491432

Metric reproduction status:
logloss     : CHECK
brier       :

In [4]:
# ============================================================
# LOG-LOSS DISCREPANCY INVESTIGATION
# ============================================================

print("SAVED LOG LOSS:", meta["test_metrics"]["logloss"])

print("\nEXACT EXTREME PROBABILITIES")

print("p = 0:", np.sum(p_test == 0))
print("p = 1:", np.sum(p_test == 1))

print(
    "Incorrect p = 0 predictions:",
    np.sum((p_test == 0) & (y_test == 1)),
)

print(
    "Incorrect p = 1 predictions:",
    np.sum((p_test == 1) & (y_test == 0)),
)

print("\nLOG-LOSS UNDER DIFFERENT CLIPPING VALUES")

for eps in [1e-3, 1e-5, 1e-7, 1e-10, 1e-15]:
    clipped = np.clip(p_test, eps, 1 - eps)

    score = log_loss(
        y_test,
        clipped,
        labels=[0, 1],
    )

    print(f"eps={eps:<10} logloss={score:.8f}")


# ============================================================
# BASELINE RESULTS
# ============================================================

print("\nBASELINE METRICS")

for name, value in baseline_scores.items():
    print(f"{name:12s}: {value:.8f}")

SAVED LOG LOSS: 0.32171693495108317

EXACT EXTREME PROBABILITIES
p = 0: 16833
p = 1: 6469
Incorrect p = 0 predictions: 4
Incorrect p = 1 predictions: 15

LOG-LOSS UNDER DIFFERENT CLIPPING VALUES
eps=0.001      logloss=0.32077343
eps=1e-05      logloss=0.32133648
eps=1e-07      logloss=0.32209904
eps=1e-10      logloss=0.32324589
eps=1e-15      logloss=0.32515743

BASELINE METRICS
logloss     : 0.69233922
brier       : 0.24959606
auc         : 0.50000000


In [5]:
# ============================================================
# COMPARE PRE-TERMINAL AND PRODUCTION PROBABILITIES
# ============================================================

# Raw XGBoost probabilities
raw_test = clf.predict_proba(te[__import__(
    "cricketwp.features",
    fromlist=["CHASE_FEATURES"]
).CHASE_FEATURES])[:, 1]

# Isotonic-calibrated probabilities BEFORE terminal rules
p_calibrated = np.interp(raw_test, iso_x, iso_y)

# Production probabilities AFTER terminal rules
p_production = p_test

# Compare metrics
print("=" * 65)
print("LOG-LOSS INVESTIGATION")
print("=" * 65)

for name, probabilities in [
    ("Raw XGBoost", raw_test),
    ("Isotonic calibrated", p_calibrated),
    ("Production (terminal rules)", p_production),
]:
    print(f"\n{name}")
    print("Log loss:", log_loss(y_test, probabilities, labels=[0, 1]))
    print("Brier:", brier_score_loss(y_test, probabilities))
    print("AUC:", roc_auc_score(y_test, probabilities))
    print("Exact zeros:", np.sum(probabilities == 0))
    print("Exact ones:", np.sum(probabilities == 1))


# ============================================================
# IDENTIFY TERMINAL RULE EFFECT
# ============================================================

changed = ~np.isclose(
    p_calibrated,
    p_production,
    atol=0,
    rtol=0,
)

print("\n" + "=" * 65)
print("TERMINAL RULE IMPACT")
print("=" * 65)

print("Rows changed by terminal rules:", changed.sum())

print(
    "Incorrect predictions among changed rows:",
    np.sum((p_production[changed] != y_test[changed]))
)

print(
    "Saved log loss:",
    meta["test_metrics"]["logloss"],
)

LOG-LOSS INVESTIGATION

Raw XGBoost
Log loss: 0.32087546586990356
Brier: 0.10162685811519623
AUC: 0.9377810022424455
Exact zeros: 0
Exact ones: 0

Isotonic calibrated
Log loss: 0.32559434489228156
Brier: 0.10290673897060984
AUC: 0.9376260397991222
Exact zeros: 16754
Exact ones: 6225

Production (terminal rules)
Log loss: 0.32540717814210646
Brier: 0.10287300958688492
AUC: 0.9376797880767507
Exact zeros: 16833
Exact ones: 6469

TERMINAL RULE IMPACT
Rows changed by terminal rules: 323
Incorrect predictions among changed rows: 0
Saved log loss: 0.32171693495108317


In [6]:
from cricketwp.eval import evaluate

print("SAVED META METRICS")
print(meta["test_metrics"])

print("\nPRODUCTION PREDICTIONS — ORIGINAL EVALUATOR")
production_scores = evaluate(y_test, p_test)

for metric, value in production_scores.items():
    saved = meta["test_metrics"].get(metric)
    difference = None if saved is None else value - saved

    print(
        f"{metric:10s} "
        f"reproduced={value:.10f} "
        f"saved={saved} "
        f"difference={difference}"
    )

print("\nRAW XGBOOST + TERMINAL RULES — ORIGINAL EVALUATOR")
raw_scores = evaluate(y_test, p_raw_production)

for metric, value in raw_scores.items():
    print(f"{metric:10s} {value:.10f}")

SAVED META METRICS
{'logloss': 0.32171693495108317, 'brier': 0.10287300925503678, 'auc': 0.9376797880767507, 'ece': 0.03188359776016573}

PRODUCTION PREDICTIONS — ORIGINAL EVALUATOR
logloss    reproduced=0.3217169350 saved=0.32171693495108317 difference=0.0
brier      reproduced=0.1028730093 saved=0.10287300925503678 difference=0.0
auc        reproduced=0.9376797881 saved=0.9376797880767507 difference=0.0
ece        reproduced=0.0318835978 saved=0.03188359776016573 difference=0.0

RAW XGBOOST + TERMINAL RULES — ORIGINAL EVALUATOR


NameError: name 'p_raw_production' is not defined

In [7]:
# Build 10 approximately equal-sized probability groups
t = pd.DataFrame({
    "p": p_test,
    "y": y_test
})

t["bin"] = pd.qcut(
    t["p"],
    q=10,
    duplicates="drop"
)

g = (
    t.groupby("bin", observed=True)
    .agg(
        pred=("p", "mean"),
        actual=("y", "mean"),
        n=("y", "size")
    )
    .reset_index()
)

# Remove pandas Interval objects; retain chart-ready values
trust_bins = [
    {
        "pred": round(float(row.pred), 6),
        "actual": round(float(row.actual), 6),
        "n": int(row.n)
    }
    for row in g.itertuples()
]

print("RELIABILITY BINS")
for row in trust_bins:
    print(row)

print("\nTOTAL STATES:", sum(row["n"] for row in trust_bins))
print("TOTAL MATCHES:", int(te["match_id"].nunique()))
print("NUMBER OF BINS:", len(trust_bins))

RELIABILITY BINS
{'pred': 0.005344, 'actual': 0.003003, 'n': 22976}
{'pred': 0.074377, 'actual': 0.057724, 'n': 11555}
{'pred': 0.16644, 'actual': 0.186331, 'n': 13417}
{'pred': 0.302443, 'actual': 0.369729, 'n': 9296}
{'pred': 0.469509, 'actual': 0.574116, 'n': 12440}
{'pred': 0.70489, 'actual': 0.772608, 'n': 13712}
{'pred': 0.857832, 'actual': 0.888943, 'n': 8158}
{'pred': 0.960074, 'actual': 0.963926, 'n': 13112}
{'pred': 0.997312, 'actual': 0.995703, 'n': 9774}

TOTAL STATES: 114440
TOTAL MATCHES: 1133
NUMBER OF BINS: 9


In [8]:
import json

from cricketwp.eval import evaluate

# 1. Evaluate the production model using the original evaluator
model_scores = evaluate(y_test, p_test)

# 2. Build a simple baseline from training-set win frequency
baseline_probability = float(train["label"].mean())

baseline_predictions = np.full(
    len(y_test),
    baseline_probability
)

baseline_scores = evaluate(y_test, baseline_predictions)

# 3. Assemble the trust report
trust = {
    "version": meta["version"],
    "evaluation_date": meta["date"],
    "metrics": model_scores,
    "baseline": baseline_scores,
    "baseline_definition": (
        "Constant probability equal to the training-set chase-win frequency"
    ),
    "bins": trust_bins,
    "n_states": int(len(te)),
    "n_matches": int(te["match_id"].nunique()),
    "known_limitations": [
        "Metrics are calculated over ball-by-ball match states, not independent matches.",
        "States from the same match are correlated.",
        "Reliability bins use quantile grouping; duplicate probability boundaries reduce the number of bins.",
        "Shortened-innings handling limitation F-01 remains unresolved."
    ]
}

# 4. Save the artifact
trust_path = ROOT / "artifacts" / "trust.json"
trust_path.parent.mkdir(parents=True, exist_ok=True)

with open(trust_path, "w", encoding="utf-8") as f:
    json.dump(trust, f, indent=2)

# 5. Read it back and verify
with open(trust_path, encoding="utf-8") as f:
    verified_trust = json.load(f)

print("SAVED:", trust_path)
print("FILE SIZE:", trust_path.stat().st_size, "bytes")
print("MODEL METRICS:", verified_trust["metrics"])
print("BASELINE METRICS:", verified_trust["baseline"])
print("RELIABILITY BINS:", len(verified_trust["bins"]))
print("TEST STATES:", verified_trust["n_states"])
print("TEST MATCHES:", verified_trust["n_matches"])

SAVED: c:\Users\karne_dvob0bk\cric-win-pred\artifacts\trust.json
FILE SIZE: 1632 bytes
MODEL METRICS: {'logloss': 0.32171693495108317, 'brier': 0.10287300925503678, 'auc': 0.9376797880767507, 'ece': 0.03188359776016573}
BASELINE METRICS: {'logloss': 0.6923392213296289, 'brier': 0.24959605633460477, 'auc': 0.5, 'ece': 0.019289651438623323}
RELIABILITY BINS: 9
TEST STATES: 114440
TEST MATCHES: 1133


In [11]:
print("LOGISTIC REGRESSION BASELINE")
print("Logistic log loss:", log_loss(test_df["label"], test_df["p_base"]))
print("XGBoost log loss:", log_loss(test_df["label"], test_df["p_model"]))
print("Test matches:", test_df["match_id"].nunique())
print("Test states:", len(test_df))

LOGISTIC REGRESSION BASELINE
Logistic log loss: 0.31091391245588657
XGBoost log loss: 0.32540717814210646
Test matches: 1133
Test states: 114440


In [12]:
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss

def bootstrap(df, n=500, seed=42):
    rng = np.random.default_rng(seed)

    groups = {
        match_id: group
        for match_id, group in df.groupby("match_id")
    }

    ids = np.array(list(groups))
    rows = []

    for _ in range(n):
        sampled_ids = rng.choice(ids, size=len(ids), replace=True)

        sample = pd.concat(
            [groups[match_id] for match_id in sampled_ids],
            ignore_index=True
        )

        model_loss = log_loss(
            sample["label"],
            sample["p_model"],
            labels=[0, 1]
        )

        baseline_loss = log_loss(
            sample["label"],
            sample["p_base"],
            labels=[0, 1]
        )

        rows.append({
            "model": model_loss,
            "baseline": baseline_loss,
            "gain": baseline_loss - model_loss
        })

    return pd.DataFrame(rows)


boot = bootstrap(test_df, n=500, seed=42)

print("BOOTSTRAP RESULTS")
print("Mean model log loss:", boot["model"].mean())
print("Mean baseline log loss:", boot["baseline"].mean())
print("Mean gain:", boot["gain"].mean())
print("95% CI:", boot["gain"].quantile([0.025, 0.975]).tolist())

BOOTSTRAP RESULTS
Mean model log loss: 0.32562451192353703
Mean baseline log loss: 0.3113007685566619
Mean gain: -0.014323743366875199
95% CI: [-0.024896403733118706, -0.003211111679719643]


In [13]:
print("TEST DATA ALIGNMENT")
print("te shape:", te.shape)
print("test_df shape:", test_df.shape)

print("\nLABELS")
print("Same labels:", np.array_equal(
    te["label"].to_numpy(),
    test_df["label"].to_numpy()
))

print("\nMODEL PREDICTIONS")
print("p_test log loss:", log_loss(te["label"], p_test))
print("test_df p_model log loss:", log_loss(
    test_df["label"], test_df["p_model"]
))

print("\nPREDICTION DIFFERENCE")
print("Max absolute difference:",
      np.max(np.abs(p_test - test_df["p_model"])))

TEST DATA ALIGNMENT
te shape: (114440, 42)
test_df shape: (114440, 4)

LABELS
Same labels: True

MODEL PREDICTIONS
p_test log loss: 0.32540717814210646
test_df p_model log loss: 0.32540717814210646

PREDICTION DIFFERENCE
Max absolute difference: 0.0


In [14]:
import json
from pathlib import Path

trust_path = Path("../artifacts/trust.json")

with open(trust_path, "r") as f:
    trust = json.load(f)

print(json.dumps(trust, indent=2))

{
  "version": "1.0.0",
  "evaluation_date": "2026-11-09",
  "metrics": {
    "logloss": 0.32171693495108317,
    "brier": 0.10287300925503678,
    "auc": 0.9376797880767507,
    "ece": 0.03188359776016573
  },
  "baseline": {
    "logloss": 0.6923392213296289,
    "brier": 0.24959605633460477,
    "auc": 0.5,
    "ece": 0.019289651438623323
  },
  "baseline_definition": "Constant probability equal to the training-set chase-win frequency",
  "bins": [
    {
      "pred": 0.005344,
      "actual": 0.003003,
      "n": 22976
    },
    {
      "pred": 0.074377,
      "actual": 0.057724,
      "n": 11555
    },
    {
      "pred": 0.16644,
      "actual": 0.186331,
      "n": 13417
    },
    {
      "pred": 0.302443,
      "actual": 0.369729,
      "n": 9296
    },
    {
      "pred": 0.469509,
      "actual": 0.574116,
      "n": 12440
    },
    {
      "pred": 0.70489,
      "actual": 0.772608,
      "n": 13712
    },
    {
      "pred": 0.857832,
      "actual": 0.888943,
      "n": 

In [15]:
print("CURRENT NOTEBOOK MODEL")
print("Best iteration:", clf.best_iteration)
print("Configured estimators:", clf.n_estimators)
print("Actual boosted rounds:", clf.get_booster().num_boosted_rounds())

print("\nCALIBRATION")
print("iso_x length:", len(iso_x))
print("iso_y length:", len(iso_y))

print("\nCURRENT PREDICTION")
print("Log loss:", log_loss(te["label"], p_test))

CURRENT NOTEBOOK MODEL
Best iteration: 121
Configured estimators: None
Actual boosted rounds: 172

CALIBRATION
iso_x length: 176
iso_y length: 176

CURRENT PREDICTION
Log loss: 0.32540717814210646


In [16]:
import json

with open("../artifacts/meta.json", "r") as f:
    meta = json.load(f)

print(json.dumps(meta, indent=2))

{
  "version": "1.0.0",
  "date": "2026-11-09",
  "chase_features": [
    "runs_needed",
    "balls_left",
    "wickets_in_hand",
    "crr",
    "rrr",
    "rrr_minus_crr",
    "target",
    "runs_last12",
    "wkts_last12",
    "dot_pct_last12",
    "partnership_runs",
    "venue_avg_first",
    "venue_chase_rate",
    "elo_diff",
    "target_vs_venue",
    "phase"
  ],
  "test_metrics": {
    "logloss": 0.32171693495108317,
    "brier": 0.10287300925503678,
    "auc": 0.9376797880767507,
    "ece": 0.03188359776016573
  }
}


In [17]:
import xgboost as xgb
import numpy as np
from sklearn.metrics import log_loss
from pathlib import Path

# Load the exact saved production model
saved_clf = xgb.XGBClassifier()
saved_clf.load_model("../artifacts/chase_xgb.json")

# Generate predictions using the same calibration arrays
saved_p = chase_curve(
    te,
    saved_clf,
    iso_x,
    iso_y
)["p_chase"].to_numpy()

print("SAVED PRODUCTION MODEL")
print("Boosted rounds:", saved_clf.get_booster().num_boosted_rounds())
print("Log loss:", log_loss(te["label"], saved_p))

print("\nNOTEBOOK MODEL")
print("Boosted rounds:", clf.get_booster().num_boosted_rounds())
print("Log loss:", log_loss(te["label"], p_test))

print("\nPREDICTION COMPARISON")
print("Maximum absolute difference:",
      np.max(np.abs(saved_p - p_test)))
print("Mean absolute difference:",
      np.mean(np.abs(saved_p - p_test)))
print("Predictions identical:",
      np.allclose(saved_p, p_test))

SAVED PRODUCTION MODEL
Boosted rounds: 172
Log loss: 0.32540717814210646

NOTEBOOK MODEL
Boosted rounds: 172
Log loss: 0.32540717814210646

PREDICTION COMPARISON
Maximum absolute difference: 0.0
Mean absolute difference: 0.0
Predictions identical: True


In [18]:
import json
import numpy as np

with open("../artifacts/chase_iso.json", "r") as f:
    saved_iso = json.load(f)

print("SAVED CALIBRATION KEYS:")
print(saved_iso.keys())

print("\nNOTEBOOK CALIBRATION:")
print("iso_x:", len(iso_x))
print("iso_y:", len(iso_y))

print("\nSAVED CALIBRATION:")
print(json.dumps(saved_iso, indent=2)[:1500])

SAVED CALIBRATION KEYS:
dict_keys(['x', 'y'])

NOTEBOOK CALIBRATION:
iso_x: 176
iso_y: 176

SAVED CALIBRATION:
{
  "x": [
    0.0017963707214221358,
    0.029738973826169968,
    0.029740329831838608,
    0.03663221746683121,
    0.03664499148726463,
    0.041462454944849014,
    0.04146356135606766,
    0.041962843388319016,
    0.04196706786751747,
    0.043688759207725525,
    0.043700333684682846,
    0.0452975332736969,
    0.04530801996588707,
    0.04669783264398575,
    0.04671819135546684,
    0.04729045182466507,
    0.04730116203427315,
    0.05421242117881775,
    0.054216139018535614,
    0.05684758722782135,
    0.056870050728321075,
    0.08135202527046204,
    0.08136239647865295,
    0.08453314751386642,
    0.08455990999937057,
    0.08607088774442673,
    0.08608051389455795,
    0.11808416247367859,
    0.11808764189481735,
    0.11875007301568985,
    0.11876309663057327,
    0.12019156664609909,
    0.12019693851470947,
    0.13024963438510895,
    0.1302611678838

In [19]:
import numpy as np
import json
from sklearn.metrics import log_loss

with open("../artifacts/chase_iso.json", "r") as f:
    saved_iso = json.load(f)

saved_x = np.array(saved_iso["x"])
saved_y = np.array(saved_iso["y"])

print("CALIBRATION COMPARISON")
print("X arrays identical:", np.array_equal(saved_x, iso_x))
print("Y arrays identical:", np.array_equal(saved_y, iso_y))
print("Maximum X difference:", np.max(np.abs(saved_x - iso_x)))
print("Maximum Y difference:", np.max(np.abs(saved_y - iso_y)))

# Re-evaluate using the saved calibration
saved_calibrated_p = chase_curve(
    te, saved_clf, saved_x, saved_y
)["p_chase"].to_numpy()

print("\nSAVED CALIBRATION EVALUATION")
print("Log loss:", log_loss(te["label"], saved_calibrated_p))
print("Max prediction difference from notebook:",
      np.max(np.abs(saved_calibrated_p - p_test)))


CALIBRATION COMPARISON
X arrays identical: True
Y arrays identical: True
Maximum X difference: 0.0
Maximum Y difference: 0.0

SAVED CALIBRATION EVALUATION
Log loss: 0.32540717814210646
Max prediction difference from notebook: 0.0


In [20]:
print("TEST DATA SUMMARY")
print("Shape:", te.shape)
print("Unique matches:", te["match_id"].nunique())
print("Date range:", te["date"].min(), "to", te["date"].max())

print("\nLABEL DISTRIBUTION")
print(te["label"].value_counts(normalize=True))

print("\nFEATURE CHECK")
print(te[CHASE_FEATURES].describe().round(4).T)

print("\nMISSING VALUES")
print(te[CHASE_FEATURES].isna().sum().sum())

TEST DATA SUMMARY
Shape: (114440, 42)
Unique matches: 1133
Date range: 2025-01-02 00:00:00 to 2026-10-25 00:00:00

LABEL DISTRIBUTION
label
0    0.527857
1    0.472143
Name: proportion, dtype: float64

FEATURE CHECK
                     count      mean      std       min       25%       50%  \
runs_needed       114440.0   96.7344  55.3084   -5.0000   54.0000   93.0000   
balls_left        114440.0   66.0952  32.5547    0.0000   40.0000   68.0000   
wickets_in_hand   114440.0    7.2031   2.3860    0.0000    6.0000    8.0000   
crr               114440.0    7.6479   3.0631    0.0000    5.7931    7.4667   
rrr               114440.0   12.3962  28.9947    0.0000    6.1463    8.7879   
rrr_minus_crr     114440.0    4.7483  29.4044  -43.7143   -2.2884    1.3737   
target            114440.0  164.8638  44.7763   18.0000  135.0000  165.0000   
runs_last12       114440.0   13.8739   7.3912    0.0000    9.0000   13.0000   
wkts_last12       114440.0    0.6335   0.7821    0.0000    0.0000    0.00